# XGBoost model

Error of the XGBoost model on the validation period, compared with the
7-day naive forecast and the RTE day-ahead forecast.

In [2]:
import pandas as pd

from load_forecast.evaluation.backtest import build_forecasts, summarize
from load_forecast.evaluation.baselines import rte_day_ahead, seasonal_naive
from load_forecast.evaluation.data import load_actuals, load_temperature
from load_forecast.evaluation.protocol import TRAIN_END, VALIDATION, target_days
from load_forecast.model import build_training_data, make_forecast_fn, train_model

In [10]:
actuals = load_actuals()
temperature = load_temperature()

consumption = actuals["consumption_mw"]
rte_forecast = actuals["forecast_d1_mw"]
X, y = build_training_data(actuals, temperature, TRAIN_END)
model = train_model(X, y)

In [11]:
def naive_7(slots):
    return seasonal_naive(consumption, slots, 7)


def rte(slots):
    return rte_day_ahead(rte_forecast, slots)


xgboost = make_forecast_fn(model, temperature)

methods = {"Naive 7 days": naive_7, "RTE day ahead": rte, "XGBoost forecast": xgboost}

In [ ]:
validation = target_days(*VALIDATION)
results = {}
for method_name, method in methods.items():
    results[method_name] = build_forecasts(method, consumption, validation)

{'Naive 7 days':                            target_day  forecast_mw  actual_mw  horizon_h
2024-08-31 22:00:00+00:00  2024-09-01      41021.0    41607.0       12.0
2024-08-31 22:30:00+00:00  2024-09-01      39834.0    40249.0       12.5
2024-08-31 23:00:00+00:00  2024-09-01      37313.0    37852.0       13.0
2024-08-31 23:30:00+00:00  2024-09-01      36582.0    37341.0       13.5
2024-09-01 00:00:00+00:00  2024-09-01      35917.0    36538.0       14.0
...                               ...          ...        ...        ...
2025-08-31 19:30:00+00:00  2025-08-31      40866.0    41625.0       33.5
2025-08-31 20:00:00+00:00  2025-08-31      40174.0    40527.0       34.0
2025-08-31 20:30:00+00:00  2025-08-31      40646.0    40983.0       34.5
2025-08-31 21:00:00+00:00  2025-08-31      41389.0    41824.0       35.0
2025-08-31 21:30:00+00:00  2025-08-31      40577.0    40875.0       35.5

[17520 rows x 4 columns], 'RTE day ahead':                            target_day  forecast_mw  actual_mw  

In [16]:
rows = []
for (method_name), forecasts in results.items():
    scores = summarize(forecasts)
    scores["method_name"] = method_name
    rows.append(scores)

overall = pd.concat(rows).set_index("method_name")
overall.round(2)

,mae_mw,mape_pct
method_name,,
Naive 7 days,3378.53,6.35
RTE day ahead,1309.20,2.63
XGBoost forecast,2850.58,5.72


In [19]:
for method_name, forecasts in results.items():
    biais = (forecasts["forecast_mw"] - forecasts["actual_mw"]).mean()
    print(method_name, round(biais))

Naive 7 days 9
RTE day ahead -980
XGBoost forecast 2138


## Conclusion

Without any hyperparameter tuning, the XGBoost model beats the 7-day
naive forecast on the validation period (MAPE 5.72 % against 6.35 %),
but remains far from the RTE forecast (2.63 %).

Its bias is +2,138 MW: the model overestimates consumption on average,
and this offset accounts for most of its error (MAE 2,851 MW). The likely
explanation is the long-term decline in consumption seen during the data
exploration: the model has no feature telling it the year, so it learns
the average level of 2012 to 2024, which is higher than the level of
2024 and 2025.